In [1]:
!pip install llm-feature-gen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.8 MB/s eta 0:00:00


In [2]:
import os
import shutil

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from llm_feature_gen.discover import discover_features_from_texts
from llm_feature_gen.generate import generate_features_from_texts
from llm_feature_gen.providers.local_provider import LocalProvider

In [3]:
import zipfile

with zipfile.ZipFile("fileDataset.zip", "r") as zip_ref:
    zip_ref.extractall(".")

FileNotFoundError: [Errno 2] No such file or directory: 'fileDataset.zip'

In [ ]:
provider = LocalProvider()
discovered = discover_features_from_texts(
    texts_or_file="./overview",
    as_set=True,
    provider=provider,
)

discovered

Features saved to outputs/discovered_text_features.json


{'proposed_features': [{'feature': 'narrative_coherence',
   'description': "Measures the logical flow and sentence structure of the description. One group tends to produce fragmented, telegraphic lists of observations (e.g., 'Dog chases squirrel. Bird on tree.'), while the other constructs connected sentences with conjunctions and temporal markers (e.g., 'The dog is chasing the squirrel which is climbing the tree').",
   'possible_values': ['fragmented_list',
    'telegraphic_utterances',
    'simple_sentences',
    'connected_narrative',
    'complex_discourse']},
  {'feature': 'uncertainty_markers',
   'description': "The frequency of expressions indicating doubt, hesitation, or lack of confidence in the observation (e.g., 'maybe', 'probably', 'I don't know', 'looks like'). This often correlates with a more spontaneous, unpolished reporting style versus a confident, definitive listing.",
   'possible_values': ['none', 'rare', 'occasional', 'frequent', 'dominant']},
  {'feature': 'ac

In [ ]:
generate_features_from_texts(
    root_folder="train",
    discovered_features_path="outputs/discovered_text_features.json",
    output_dir="outputs/train_features",
    merge_to_single_csv=True,
    provider=provider,
)


positive: 100%|██████████| 70/70 [02:32<00:00,  2.18s/file]


{'negative': 'outputs/train_features/negative_feature_values.csv',
 'positive': 'outputs/train_features/positive_feature_values.csv',
 '__merged__': 'outputs/train_features/all_feature_values.csv'}

In [ ]:
train_csv = "outputs/train_features/all_feature_values.csv"

train_df = pd.read_csv(train_csv)

print(train_df.shape)
print(train_df["Class"].value_counts())
train_df.head()

(241, 13)
Class
negative    171
positive     70
Name: count, dtype: int64


,File,Class,narrative_coherence,uncertainty_markers,action_verb_tense_consistency,entity_specificity_level,self_referential_metacommentary,spatial_organization_pattern,emotional_interpretation_inference,lexical_variation_and_repetition,quantification_precision,discourse_marker_usage,raw_llm_output
0,100tr0.txt,negative,simple_sentences,occasional,strict_present,basic_attributes,minimal,random_jump,purely_physical,moderate_variation,estimates_only,sparse,"{""features"": {""narrative_coherence"": ""simple_s..."
1,101tr0.txt,negative,connected_narrative,occasional,mixed_tenses,basic_attributes,minimal,random_jump,purely_physical,moderate_variation,estimates_only,frequent,"{""features"": {""narrative_coherence"": ""connecte..."
2,102tr0.txt,negative,connected_narrative,occasional,strict_present,basic_attributes,absent,systematic_scan,minimal_inference,moderate_variation,estimates_only,sparse,"{""features"": {""narrative_coherence"": ""connecte..."
3,103tr0.txt,negative,simple_sentences,frequent,mixed_tenses,descriptive_details,minimal,random_jump,moderate_inference,low_variation,counting_with_hesitation,moderate,"{""features"": {""narrative_coherence"": ""simple_s..."
4,104tr0.txt,negative,connected_narrative,frequent,strict_present,basic_attributes,moderate,random_jump,minimal_inference,low_variation,estimates_only,moderate,"{""features"": {""narrative_coherence"": ""connecte..."


In [ ]:
generate_features_from_texts(
    root_folder="test",
    discovered_features_path="outputs/discovered_text_features.json",
    output_dir="outputs/test_features",
    merge_to_single_csv=True,
    provider=provider,
)

positive: 100%|██████████| 19/19 [00:27<00:00,  1.43s/file]


{'negative': 'outputs/test_features/negative_feature_values.csv',
 'positive': 'outputs/test_features/positive_feature_values.csv',
 '__merged__': 'outputs/test_features/all_feature_values.csv'}

In [ ]:
test_csv = "outputs/test_features/all_feature_values.csv"

test_df = pd.read_csv(test_csv)

print(test_df.shape)
print(test_df["Class"].value_counts())
test_df

(61, 13)
Class
negative    42
positive    19
Name: count, dtype: int64


,File,Class,narrative_coherence,uncertainty_markers,action_verb_tense_consistency,entity_specificity_level,self_referential_metacommentary,spatial_organization_pattern,emotional_interpretation_inference,lexical_variation_and_repetition,quantification_precision,discourse_marker_usage,raw_llm_output
0,10te0.txt,negative,fragmented_list,frequent,mixed_tenses,basic_attributes,frequent,random_jump,minimal_inference,low_variation,estimates_only,moderate,"{""features"": {""narrative_coherence"": ""fragment..."
1,11te0.txt,negative,simple_sentences,occasional,strict_present,basic_attributes,absent,random_jump,purely_physical,moderate_variation,no_counting,none,"{""features"": {""narrative_coherence"": ""simple_s..."
2,12te0.txt,negative,fragmented_list,none,strict_present,descriptive_details,absent,random_jump,purely_physical,moderate_variation,exact_immediate,none,"{""features"": {""narrative_coherence"": ""fragment..."
3,13te0.txt,negative,simple_sentences,occasional,mixed_tenses,basic_attributes,moderate,random_jump,purely_physical,high_repetition,estimates_only,sparse,"{""features"": {""narrative_coherence"": ""simple_s..."
4,14te0.txt,negative,connected_narrative,frequent,strict_present,basic_attributes,absent,random_jump,minimal_inference,low_variation,exact_immediate,frequent,"{""features"": {""narrative_coherence"": ""connecte..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,5te1.txt,positive,fragmented_list,occasional,mixed_tenses,basic_attributes,moderate,random_jump,minimal_inference,high_repetition,no_counting,frequent,"{""features"": {""narrative_coherence"": ""fragment..."
57,6te1.txt,positive,fragmented_list,frequent,mixed_tenses,basic_attributes,absent,random_jump,purely_physical,low_variation,estimates_only,sparse,"{""features"": {""narrative_coherence"": ""fragment..."
58,7te1.txt,positive,connected_narrative,occasional,strict_present,basic_attributes,absent,random_jump,minimal_inference,moderate_variation,estimates_only,sparse,"{""features"": {""narrative_coherence"": ""connecte..."
59,8te1.txt,positive,connected_narrative,occasional,strict_present,basic_attributes,minimal,random_jump,purely_physical,low_variation,estimates_only,frequent,"{""features"": {""narrative_coherence"": ""connecte..."


In [ ]:
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain classes:")
print(train_df["Class"].value_counts())

print("\nTest classes:")
print(test_df["Class"].value_counts())

Train rows: 241
Test rows: 61

Train classes:
Class
negative    171
positive     70
Name: count, dtype: int64

Test classes:
Class
negative    42
positive    19
Name: count, dtype: int64


In [ ]:
train_files = set(train_df["File"].astype(str))
test_files = set(test_df["File"].astype(str))

overlap = train_files.intersection(test_files)

print("Počet shodných názvů souborů ve train a test:", len(overlap))

if overlap:
    print("Příklady překryvu:")
    print(list(overlap)[:10])

Počet shodných názvů souborů ve train a test: 0


In [ ]:
ignored_columns = {"File", "Class", "raw_llm_output"}

feature_columns = [
    column for column in train_df.columns
    if column not in ignored_columns
]

X_train = train_df[feature_columns].copy()
y_train = train_df["Class"].astype(str)

X_test = test_df[feature_columns].copy()
y_test = test_df["Class"].astype(str)

for column in feature_columns:
    X_train[column] = (
        X_train[column]
        .astype("string")
        .fillna("missing")
        .str.strip()
        .replace("", "missing")
    )

    X_test[column] = (
        X_test[column]
        .astype("string")
        .fillna("missing")
        .str.strip()
        .replace("", "missing")
    )

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

In [ ]:
label_encoder = LabelEncoder()

y_train_xgb = label_encoder.fit_transform(y_train)
y_test_xgb = label_encoder.transform(y_test)

print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'negative': np.int64(0), 'positive': np.int64(1)}


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            feature_columns,
        )
    ],
    remainder="drop",
)

classifier = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
)

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ]
)

xgb_model.fit(X_train, y_train_xgb)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['narrative_coherence',
                                                   'uncertainty_markers',
                                                   'action_verb_tense_consistency',
                                                   'entity_specificity_level',
                                                   'self_referential_metacommentary',
                                                   'spatial_organization_pattern',
                                                   'emotional_interpretation_inference',
                                                   'lexical_variation_and_rep...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [ ]:
classes = ["positive", "negative"]

In [ ]:
xgb_predictions_num = xgb_model.predict(X_test)
xgb_predictions = label_encoder.inverse_transform(xgb_predictions_num)

print(classification_report(y_test, xgb_predictions, digits=4))

cm = confusion_matrix(y_test, xgb_predictions, labels=classes)

pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in classes],
    columns=[f"pred_{label}" for label in classes],
)

              precision    recall  f1-score   support

    negative     0.7556    0.8095    0.7816        42
    positive     0.5000    0.4211    0.4571        19

    accuracy                         0.6885        61
   macro avg     0.6278    0.6153    0.6194        61
weighted avg     0.6760    0.6885    0.6805        61



,pred_positive,pred_negative
true_positive,8,11
true_negative,8,34
